# 🐝 아크릴 격자 합성 사진 만들기

찍어 둔 사진·영상 위에 **서현 설계 아크릴 격자**(투명 2T, 구멍 3.0 mm, 간격 4.4 mm, 37개, 마킹실 안지름 33 mm)를 그려 넣어요.
**미리 연습용**이에요 — 진짜 격자가 오면 진짜 사진으로 다시 공부시켜야 해요.

## 사용법 — 코드는 안 고쳐도 돼요
1. 드라이브 `hornet_finetune` 안에 폴더 2개 만들기
   - **`synth_src`** → 흰 종이에서 찍은(격자 없는) Roboflow zip 1개 넣기
   - **`synth_test_src`** → 격자 없이 찍은 **시험용** 영상 넣기 (말벌을 화면 **가운데**에 두고 찍은 것 + 빈 종이 영상)
2. 위 메뉴 **런타임 → 모두 실행** (GPU 필요 없음) → 드라이브 **허용**
3. 끝나면
   - `hornet_finetune/acrylic_synth.zip` → 공부 노트북에 쓰는 zip (가장 새 zip이라 자동으로 쓰임)
   - `hornet_finetune/test/acrylic_○○.mp4` → 시험 노트북에 쓰는 영상

## 1. 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob, zipfile, shutil, random, pathlib, yaml, cv2, numpy as np

DRIVE_DIR = '/content/drive/MyDrive/hornet_finetune'
SRC_DIR   = f'{DRIVE_DIR}/synth_src'        # 격자 없는 Roboflow zip
VID_SRC   = f'{DRIVE_DIR}/synth_test_src'   # 격자 없는 시험 영상
OUT_ZIP   = f'{DRIVE_DIR}/acrylic_synth.zip'
TEST_DIR  = f'{DRIVE_DIR}/test'

ASPECT    = 16 / 9   # Roboflow가 16:9 영상을 정사각형(640×640)으로 늘린 비율
HORNET_MM = 22       # 네모 긴 변 ≈ 말벌 몸길이 + 다리 (mm) → 1 mm가 몇 화소인지 계산
BODY_MM   = 20       # 영상에서 어두운 몸통 길이 (mm) → 시험 영상 격자 크기를 말벌에 맞춤
VARIANTS  = 2        # 사진 1장마다 격자 사진 몇 장 만들지
KEEP_ORIGINAL = True # 원래 사진도 같이 넣기
PITCH, HOLE, R_IN, PLATE = 4.4, 3.0, 16.5, 38.0   # 서현 설계값 (mm)
SEED = 0

## 2. 격자 그리는 방법 (건드리지 마세요)

In [ ]:
HOLES = [(i * PITCH, j * PITCH) for i in range(-3, 4) for j in range(-3, 4)
         if not ((abs(i) == 3 and abs(j) >= 2) or (abs(j) == 3 and abs(i) >= 2))]   # 37개

def draw_acrylic(img, s, cx, cy, rng):
    """img 위에 아크릴 격자를 그림. s = 1 mm당 화소, (cx, cy) = 마킹실 가운데. 마킹실 안쪽 마스크도 돌려줌."""
    h, w = img.shape[:2]
    ang = np.deg2rad(rng.uniform(-8, 8)); ca, sa = np.cos(ang), np.sin(ang)
    P = lambda x, y: (cx + s * (x * ca - y * sa), cy + s * (x * sa + y * ca))
    sq = lambda x, y, a: np.array([P(x - a, y - a), P(x + a, y - a), P(x + a, y + a), P(x - a, y + a)], np.int32)
    out = img.astype(np.float32)
    yy, xx = np.mgrid[0:h, 0:w]
    inside = (xx - cx) ** 2 + (yy - cy) ** 2 <= (R_IN * s) ** 2
    box = (rng.uniform(150, 215) + rng.uniform(-12, 12, 3)).astype(np.float32)   # 회색~베이지
    out[~inside] = box + rng.normal(0, 4, (h, w, 1))[~inside]          # 마킹실 바깥 = 프린트 상자
    plate = np.zeros((h, w), np.uint8); cv2.fillPoly(plate, [sq(0, 0, PLATE / 2)], 1)
    holes = [sq(hx, hy, HOLE / 2) for hx, hy in HOLES]
    hole = np.zeros((h, w), np.uint8); cv2.fillPoly(hole, holes, 1)
    bars = (plate == 1) & (hole == 0)
    a = rng.uniform(0.08, 0.18)
    out[bars] = out[bars] * (1 - a) + np.array([235, 228, 205], np.float32) * a   # 투명 막대 살짝 푸르게
    lw = max(1, int(round(0.12 * s)))
    m = np.zeros((h, w), np.uint8); cv2.polylines(m, holes, True, 1, lw)
    out[m == 1] *= 1 - rng.uniform(0.2, 0.36)                          # 구멍 가장자리 선
    off = int(round(rng.uniform(0.1, 0.3) * s))
    if off:
        m2 = np.zeros((h, w), np.uint8); cv2.polylines(m2, [q + off for q in holes], True, 1, lw)
        out[m2 == 1] = out[m2 == 1] * 0.8 + 51                        # 2 mm 두께 안쪽 벽 (밝게)
    for sx, sy in [(-16, -16), (16, -16), (-16, 16), (16, 16)]:
        cv2.circle(out, tuple(int(v) for v in P(sx, sy)), int(1.7 * s), (box * 0.75).tolist(), -1)
    for _ in range(rng.integers(1, 4)):                                 # 빛 반사
        x0, y0 = rng.uniform(-PLATE / 2, PLATE / 2, 2); L = rng.uniform(4, 12); t = rng.uniform(0, np.pi)
        ov = out.copy()
        cv2.line(ov, tuple(int(v) for v in P(x0, y0)), tuple(int(v) for v in P(x0 + L * np.cos(t), y0 + L * np.sin(t))),
                 (255, 255, 255), max(1, int(0.8 * s)))
        al = rng.uniform(0.2, 0.45); out = out * (1 - al) + ov * al
    return np.clip(out, 0, 255).astype(np.uint8), inside

## 3. Roboflow 사진에 격자 그리기

In [ ]:
zips = sorted(glob.glob(f'{SRC_DIR}/*.zip'), key=os.path.getmtime)
if not zips:
    raise FileNotFoundError('❌ hornet_finetune/synth_src 폴더에 격자 없는 Roboflow zip을 넣고 다시 [런타임 → 모두 실행] 하세요.')
SRC = pathlib.Path('/content/synth_src'); OUT = pathlib.Path('/content/acrylic_synth')
shutil.rmtree(SRC, ignore_errors=True); shutil.rmtree(OUT, ignore_errors=True)
zipfile.ZipFile(zips[-1]).extractall(SRC)
y = yaml.safe_load(open(next(SRC.rglob('data.yaml'))))
root = next(SRC.rglob('data.yaml')).parent
names = y['names']; names = [names[k] for k in sorted(names)] if isinstance(names, dict) else list(names)
print('원본:', os.path.basename(zips[-1]), '| 클래스:', names)

def read_labels(p):
    return [list(map(float, l.split()[:5])) for l in p.read_text().strip().splitlines() if l.strip()] if p.exists() else []

items = []
for split in ['train', 'valid', 'test']:
    for ip in sorted((root / split / 'images').glob('*')):
        items.append((split, ip, read_labels(root / split / 'labels' / (ip.stem + '.txt'))))

scales = []
for _, ip, labs in items:
    im = cv2.imread(str(ip)); H, W = im.shape[:2]; Hu = round(H / ASPECT)
    scales += [max(b[3] * W, b[4] * Hu) / HORNET_MM for b in labs]
S_MED = float(np.median(scales)) if scales else 6.0
print(f'1 mm ≈ {S_MED:.1f} 화소 (네모 {len(scales)}개로 계산)')

rng = np.random.default_rng(SEED); made = skipped = 0
for split, ip, labs in items:
    (OUT / split / 'images').mkdir(parents=True, exist_ok=True); (OUT / split / 'labels').mkdir(parents=True, exist_ok=True)
    im = cv2.imread(str(ip)); H, W = im.shape[:2]; Hu = round(H / ASPECT)
    if KEEP_ORIGINAL:
        shutil.copy(ip, OUT / split / 'images' / ip.name)
        (OUT / split / 'labels' / (ip.stem + '.txt')).write_text(''.join(f'{int(b[0])} {b[1]:.6f} {b[2]:.6f} {b[3]:.6f} {b[4]:.6f}\n' for b in labs))
    imu = cv2.resize(im, (W, Hu))
    s = float(np.median([max(b[3] * W, b[4] * Hu) / HORNET_MM for b in labs])) if labs else S_MED
    R = R_IN * s
    for v in range(VARIANTS):
        if labs:
            b = labs[rng.integers(len(labs))]; bx, by, bw, bh = b[1] * W, b[2] * Hu, b[3] * W, b[4] * Hu
            r = max(0.0, R - np.hypot(bw, bh) / 2) * np.sqrt(rng.uniform()); t = rng.uniform(0, 2 * np.pi)
            cx, cy = bx + r * np.cos(t), by + r * np.sin(t)
        else:
            cx, cy = W / 2 + rng.uniform(-0.2, 0.2) * W, Hu / 2 + rng.uniform(-0.2, 0.2) * Hu
        out, inside = draw_acrylic(imu, s, cx, cy, rng)
        keep, bad = [], False
        for c, x, y_, w, h in labs:
            gx, gy = np.meshgrid(np.linspace(x - w / 2, x + w / 2, 10) * W, np.linspace(y_ - h / 2, y_ + h / 2, 10) * Hu)
            frac = np.mean((gx - cx) ** 2 + (gy - cy) ** 2 <= R ** 2)
            if frac >= 0.6:
                x1, x2 = max(x - w / 2, (cx - R) / W), min(x + w / 2, (cx + R) / W)
                y1, y2 = max(y_ - h / 2, (cy - R) / Hu), min(y_ + h / 2, (cy + R) / Hu)
                keep.append((int(c), (x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1))
            elif frac > 0.15:
                bad = True
        if bad: skipped += 1; continue
        name = f'{ip.stem}_acr{v}'
        cv2.imwrite(str(OUT / split / 'images' / f'{name}.jpg'), cv2.resize(out, (W, H)))
        (OUT / split / 'labels' / f'{name}.txt').write_text(''.join(f'{c} {x:.6f} {y_:.6f} {w:.6f} {h:.6f}\n' for c, x, y_, w, h in keep))
        made += 1

yaml.safe_dump({'train': '../train/images', 'val': '../valid/images', 'test': '../test/images',
                'nc': len(names), 'names': names}, open(OUT / 'data.yaml', 'w'), allow_unicode=True)
if os.path.exists(OUT_ZIP): os.remove(OUT_ZIP)
with zipfile.ZipFile(OUT_ZIP, 'w') as z:
    for p in OUT.rglob('*'):
        if p.is_file(): z.write(p, p.relative_to(OUT))
print(f'✅ 격자 사진 {made}장 만듦 (애매해서 뺀 것 {skipped}장) → {OUT_ZIP}')

## 4. 만든 사진 눈으로 확인 (빨간 네모 = 라벨)

In [ ]:
import matplotlib.pyplot as plt
shots = sorted((OUT / 'train' / 'images').glob('*_acr*'))
shots = random.Random(1).sample(shots, k=min(8, len(shots)))
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax in axes.flat: ax.axis('off')
for ax, p in zip(axes.flat, shots):
    im = cv2.imread(str(p)); H, W = im.shape[:2]
    for c, x, y_, w, h in read_labels(OUT / 'train' / 'labels' / (p.stem + '.txt')):
        cv2.rectangle(im, (int((x - w / 2) * W), int((y_ - h / 2) * H)), (int((x + w / 2) * W), int((y_ + h / 2) * H)), (0, 0, 255), 2)
    ax.imshow(cv2.cvtColor(cv2.resize(im, (W, round(H / ASPECT))), cv2.COLOR_BGR2RGB)); ax.set_title(p.name[:30], fontsize=8)
plt.tight_layout(); plt.show()

## 5. 시험 영상에 격자 그리기 (격자는 화면 가운데에 고정)

In [ ]:
os.makedirs(VID_SRC, exist_ok=True); os.makedirs(TEST_DIR, exist_ok=True)
vids = sorted(f for f in glob.glob(f'{VID_SRC}/*') if os.path.splitext(f)[1].lower() in {'.mp4', '.mov', '.avi', '.mkv', '.wmv'})
if not vids:
    print('(synth_test_src 폴더에 영상이 없어서 건너뜀)')
firsts = []
for k, f in enumerate(vids):
    cap = cv2.VideoCapture(f); fps = cap.get(cv2.CAP_PROP_FPS) or 30
    W, H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    s0 = S_MED * W / 640                    # 학습 사진(가로 640)과 같은 높이라고 볼 때
    best = 0
    for _ in range(30):                     # 앞 30장면에서 가장 큰 어두운 덩어리(말벌) 길이 재기
        ok, fr = cap.read()
        if not ok: break
        g = cv2.GaussianBlur(cv2.cvtColor(fr, cv2.COLOR_BGR2GRAY), (5, 5), 0)
        _, th = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        cs, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cs = [c for c in cs if cv2.contourArea(c) < 0.2 * W * H]
        if cs: best = max(best, max(max(cv2.minAreaRect(c)[1]) for c in cs))
    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
    s_blob = best / BODY_MM
    s = s_blob if 0.4 * s0 < s_blob < 2.5 * s0 else s0   # 말벌 몸 크기로 맞춤 (못 찾으면 학습 기준)
    print(f'{os.path.basename(f)}: 1 mm ≈ {s:.1f} 화소 (말벌로 잰 값 {s_blob:.1f}, 학습 기준 {s0:.1f})')
    dst = f'{TEST_DIR}/acrylic_{pathlib.Path(f).stem}.mp4'
    wr = cv2.VideoWriter(dst, cv2.VideoWriter_fourcc(*'mp4v'), fps, (W, H)); n = 0
    while True:
        ok, fr = cap.read()
        if not ok: break
        out, _ = draw_acrylic(fr, s, W / 2, H / 2, np.random.default_rng(1000 + k))   # 같은 격자를 모든 장면에
        wr.write(out); n += 1
        if n == 1: firsts.append((os.path.basename(dst), out))
    wr.release(); print(f'✅ {os.path.basename(dst)} ({n}장면)')
for t, im in firsts:
    plt.figure(figsize=(8, 4.5)); plt.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); plt.title(t); plt.axis('off'); plt.show()
print('끝! 이제 공부 노트북(RUN_NAME = hornet_v4) → 시험 노트북 순서로 실행하세요.')